In [15]:
import sqlite3
import pandas as pd
import logging
import os

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/get_vendor_summary.log",
    level=logging.DEBUG
)

conn = sqlite3.connect("inventory.db")

In [16]:
def get_vendor_summary(conn):
    """
    Merge purchase, sales, and freight data
    to create the vendor summary.
    """
    
    query = """
    WITH FreightSummary AS (
        SELECT
            VendorNumber,
            SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),

    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price,
            pp.Volume
    ),

    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY
            VendorNo,
            Brand
    )

    SELECT
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC
    """
    
    return pd.read_sql_query(query, conn)

In [17]:
import os

print("Current folder:", os.getcwd())
print("Database exists:", os.path.exists("inventory.db"))
print("Database path:", os.path.abspath("inventory.db"))

Current folder: D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis\notebooks\vendor_analysis.ipynb
Database exists: True
Database path: D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis\notebooks\vendor_analysis.ipynb\inventory.db


In [18]:
conn.close()

conn = sqlite3.connect("../inventory.db")

In [19]:
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)

,name
0,begin_inventory
1,end_inventory
2,purchase_prices
3,purchases
4,sales
5,vendor_invoice
6,vendor_sales_summary


In [20]:
vendor_sales_summary = get_vendor_summary(conn)

vendor_sales_summary.head()

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.60,142049.0,5101919.51,672819.31,260999.20,68601.68
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.22,160247.0,4819073.49,561512.37,294438.66,144929.24
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.68,187140.0,4538120.60,461140.15,343854.07,123780.22
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261197.94,200412.0,4475972.88,420050.01,368242.80,257032.07
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.01,135838.0,4223107.62,545778.28,249587.83,257032.07


In [21]:
print("Rows:", len(vendor_sales_summary))
print("Columns:", len(vendor_sales_summary.columns))

Rows: 10692
Columns: 14


In [22]:
vendor_sales_summary['Volume'] = (
    vendor_sales_summary['Volume'].astype('float64')
)

vendor_sales_summary['Volume'] = (
    vendor_sales_summary['Volume'].fillna(0)
)

vendor_sales_summary['VendorName'] = (
    vendor_sales_summary['VendorName'].str.strip()
)

vendor_sales_summary['GrossProfit'] = (
    vendor_sales_summary['TotalSalesDollars']
    - vendor_sales_summary['TotalPurchaseDollars']
)

In [23]:
vendor_sales_summary['GrossProfit'].min()

-52002.780000000006

In [24]:
vendor_sales_summary['StockTurnover'] = (
    vendor_sales_summary['TotalSalesQuantity']
    / vendor_sales_summary['TotalPurchaseQuantity']
)

vendor_sales_summary['SalesToPurchaseRatio'] = (
    vendor_sales_summary['TotalSalesDollars']
    / vendor_sales_summary['TotalPurchaseDollars']
)

In [25]:
vendor_sales_summary[
    ['StockTurnover', 'SalesToPurchaseRatio']
].head()

,StockTurnover,SalesToPurchaseRatio
0,0.979108,1.338647
1,0.976890,1.266830
2,0.998575,1.327594
3,0.993703,1.372493
4,0.983556,1.396897


In [26]:
vendor_sales_summary['Description'] = (
    vendor_sales_summary['Description'].str.strip()
)

In [27]:
vendor_sales_summary['ProfitMargin'] = (
    vendor_sales_summary['GrossProfit']
    / vendor_sales_summary['TotalSalesDollars']
)

In [28]:
vendor_sales_summary[['GrossProfit', 'ProfitMargin']].head()

,GrossProfit,ProfitMargin
0,1290667.91,0.252977
1,1015032.27,0.210628
2,1119816.92,0.246758
3,1214774.94,0.271399
4,1199901.61,0.284128


In [29]:
def ingest_data(df, conn):
    """
    Ingest the cleaned vendor summary into SQLite.
    """
    df.to_sql(
        'vendor_sales_summary',
        conn,
        if_exists='replace',
        index=False
    )

In [30]:
ingest_data(vendor_sales_summary, conn)

In [31]:
pd.read_sql_query(
    "SELECT * FROM vendor_sales_summary LIMIT 5",
    conn
)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,StockTurnover,SalesToPurchaseRatio,ProfitMargin
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,5101919.51,672819.31,260999.20,68601.68,1290667.91,0.979108,1.338647,0.252977
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,4819073.49,561512.37,294438.66,144929.24,1015032.27,0.976890,1.266830,0.210628
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,4538120.60,461140.15,343854.07,123780.22,1119816.92,0.998575,1.327594,0.246758
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,4475972.88,420050.01,368242.80,257032.07,1214774.94,0.993703,1.372493,0.271399
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,4223107.62,545778.28,249587.83,257032.07,1199901.61,0.983556,1.396897,0.284128


In [32]:
print(
    pd.read_sql_query(
        "SELECT COUNT(*) AS total_rows FROM vendor_sales_summary",
        conn
    )
)

   total_rows
0       10692


In [33]:
import sqlite3
import pandas as pd

db_path = r"D:\Kuliah\Practice\Data Analyst Portfolio\Vendor Performance Data Analytics End-To-End Project  SQL + Python + Power BI + Reporting\Vendor_Analysis\inventory.db"

conn = sqlite3.connect(db_path)

In [34]:
def get_vendor_summary(conn):
    """
    Merge purchase, sales, and freight data
    to create the vendor summary.
    """
    
    query = """
    WITH FreightSummary AS (
        SELECT
            VendorNumber,
            SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),

    PurchaseSummary AS (
        SELECT
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price,
            pp.Volume
    ),

    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY
            VendorNo,
            Brand
    )

    SELECT
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ps.VendorNumber = ss.VendorNo
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC
    """
    
    return pd.read_sql_query(query, conn)

In [35]:
vendor_sales_summary = get_vendor_summary(conn)

In [36]:
vendor_sales_summary.shape

(10692, 14)

In [37]:
vendor_sales_summary['Volume'] = (
    vendor_sales_summary['Volume'].astype('float64')
)

vendor_sales_summary['Volume'] = (
    vendor_sales_summary['Volume'].fillna(0)
)

vendor_sales_summary['VendorName'] = (
    vendor_sales_summary['VendorName'].str.strip()
)

vendor_sales_summary['GrossProfit'] = (
    vendor_sales_summary['TotalSalesDollars']
    - vendor_sales_summary['TotalPurchaseDollars']
)

In [38]:
vendor_sales_summary['GrossProfit'].min()

-52002.780000000006

In [39]:
vendor_sales_summary['ProfitMargin'] = (
    vendor_sales_summary['GrossProfit']
    / vendor_sales_summary['TotalSalesDollars']
)

In [40]:
vendor_sales_summary[['GrossProfit', 'ProfitMargin']].head()

,GrossProfit,ProfitMargin
0,1290667.91,0.252977
1,1015032.27,0.210628
2,1119816.92,0.246758
3,1214774.94,0.271399
4,1199901.61,0.284128


In [41]:
vendor_sales_summary.shape

(10692, 16)

In [42]:
vendor_sales_summary.columns.tolist()

['VendorNumber',
 'VendorName',
 'Brand',
 'Description',
 'PurchasePrice',
 'ActualPrice',
 'Volume',
 'TotalPurchaseQuantity',
 'TotalPurchaseDollars',
 'TotalSalesQuantity',
 'TotalSalesDollars',
 'TotalSalesPrice',
 'TotalExciseTax',
 'FreightCost',
 'GrossProfit',
 'ProfitMargin']

In [43]:
vendor_sales_summary["StockTurnover"] = (
    vendor_sales_summary["TotalSalesQuantity"]
    / vendor_sales_summary["TotalPurchaseQuantity"]
)

vendor_sales_summary["SalesToPurchaseRatio"] = (
    vendor_sales_summary["TotalSalesDollars"]
    / vendor_sales_summary["TotalPurchaseDollars"]
)

In [45]:
vendor_sales_summary.shape

(10692, 18)